In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from google.colab import drive


drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Multimodal_Stroke_Project/processed_dataset'
CSV_PATH = os.path.join(BASE_DIR, 'master_dataset.csv')
IMG_DIR = os.path.join(BASE_DIR, 'images')

IMG_SIZE = 224 # Standard for MobileNet

print("---LOADING DATA---")
df = pd.read_csv(CSV_PATH)
df['image_path'] = df['filename'].apply(lambda x: os.path.join(IMG_DIR, x))
valid_rows = []
for idx, row in df.iterrows():
    if os.path.exists(row['image_path']):
        valid_rows.append(row)
df = pd.DataFrame(valid_rows)
print(f"Verified {len(df)} samples with valid images.")

print("\n---PREPROCESSING TABULAR DATA---")
# Handle Missing BMI (Fill with mean)
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

# Encode Categorical Variables
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Select Features for the Neural Network
features = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'] + categorical_cols
X_tab = df[features].values
y = df['stroke'].values

# Normalize Data 
scaler = StandardScaler()
X_tab = scaler.fit_transform(X_tab)

print(f"Tabular Input Shape: {X_tab.shape}")

print("\n---PREPROCESSING IMAGES---")
def load_and_process_images(image_paths):
    images = []
    for path in image_paths:
        
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) # Resize to 224x224
        images.append(img)
    return np.array(images) / 255.0 # Normalize pixel values to 0-1

X_img = load_and_process_images(df['image_path'].values)
print(f"Image Input Shape: {X_img.shape}")

# Split Data (Train/Test)
X_tab_train, X_tab_test, X_img_train, X_img_test, y_train, y_test = train_test_split(
    X_tab, X_img, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining Samples: {len(y_train)} | Testing Samples: {len(y_test)}")

print("\n---BUILDING THE Y-NETWORK---")

# BRANCH 1: Image (CNN)
input_img = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="Image_Input")
# MobileNetV2 (Pre-trained on ImageNet)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_img)
# Freeze base layers (so we don't destroy learned features)
for layer in base_model.layers:
    layer.trainable = False
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
out_img = Dropout(0.5)(x)

# BRANCH 2: Tabular (MLP)
input_tab = Input(shape=(X_tab.shape[1],), name="Tabular_Input")
y_tab = Dense(32, activation='relu')(input_tab)
y_tab = Dense(16, activation='relu')(y_tab)
out_tab = Dropout(0.3)(y_tab)

# FUSION (The Stem)
combined = Concatenate()([out_img, out_tab])
z = Dense(32, activation='relu')(combined)
z = Dense(1, activation='sigmoid', name="Risk_Output")(z) # Sigmoid for 0-1 Probability

# Compile Model
model = Model(inputs=[input_img, input_tab], outputs=z)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy', 'Recall'])

model.summary()

print("\n---TRAINING---")
history = model.fit(
    [X_img_train, X_tab_train], y_train,
    validation_data=([X_img_test, X_tab_test], y_test),
    epochs=20,
    batch_size=32
)

print("\n---SAVING---")
save_path = os.path.join(BASE_DIR, 'final_multimodal_model.h5')
model.save(save_path)
print(f"Model saved successfully at: {save_path}")
print("Ready for deployment!")

In [ ]:
import joblib

scaler_path = os.path.join(BASE_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)

print(f"Scaler saved at: {scaler_path}")